In [ ]:
%pip install chromadb

In [1]:
%pip install sentence-transformers

  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ----------------- ---------------------- 4.7/10.7 MB 22.6 MB/s eta 0:00:01
   ---------------------------------------- 10.7/10.7 MB 26.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/612.9 kB ? eta -:--:--
   --------------------------------------- 612.9/612.9 kB 23.0 MB/s eta 0:00:00
   --------------------


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
sentences = [
    "How do I read a file in Python?",
    "Open a file using the open() function",
    "Python file I/O tutorial",
    "What is a for loop?",
    "Iterating over a list in Python",
    "How to use list comprehensions",
    "Docker container vs Docker image",
    "What is a Dockerfile?",
    "Sort a list in Python",
    "The sky is blue and the sun is yellow"
]

user_query = "What is iteration?"

embeddings = model.encode(sentences)

user_embedding = model.encode(user_query)
user_similarities = model.similarity(user_embedding, embeddings)[0]

top_k = min(5, len(user_similarities))
top_results = np.argpartition(-user_similarities, range(top_k))[0:top_k]

print("top results:")
for i in top_results:
    print(f"- {sentences[i]} (score: {user_similarities[i]:.3f})")

print(embeddings)
print(user_similarities)

c:\Users\sndes\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7642.47it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


top results:
- What is a for loop? (score: 0.590)
- Iterating over a list in Python (score: 0.409)
- How to use list comprehensions (score: 0.253)
- Python file I/O tutorial (score: 0.208)
- What is a Dockerfile? (score: 0.119)
[[ 0.01545032  0.06672422 -0.10842565 ...  0.12807032  0.05996446
  -0.00023202]
 [ 0.04272832  0.05857249 -0.06462938 ...  0.03630496  0.05146503
   0.03034442]
 [-0.09018744  0.04470504 -0.04212139 ...  0.14796032  0.06810074
  -0.05837488]
 ...
 [-0.06089142  0.07327832 -0.02317577 ...  0.09920333  0.10328437
   0.08065025]
 [ 0.00752256  0.051953    0.04772322 ...  0.0143225   0.0937846
   0.06545071]
 [ 0.01770419  0.08949164  0.05973819 ...  0.02115932 -0.07152769
   0.04811272]]
tensor([0.0810, 0.0512, 0.2078, 0.5901, 0.4086, 0.2528, 0.0207, 0.1187, 0.0763,
        0.0594])


In [3]:
from sentence_transformers.cross_encoder import CrossEncoder

model_cross = CrossEncoder("cross-encoder/stsb-distilroberta-base")


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4403.45it/s]
RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
ranks = model_cross.rank(user_query, sentences)
for rank in ranks:
    print(f"{rank['score']:.3f} - {sentences[rank['corpus_id']]}")


0.423 - What is a for loop?
0.358 - Iterating over a list in Python
0.095 - What is a Dockerfile?
0.049 - Sort a list in Python
0.036 - Open a file using the open() function
0.029 - How to use list comprehensions
0.019 - Docker container vs Docker image
0.012 - How do I read a file in Python?
0.010 - Python file I/O tutorial
0.009 - The sky is blue and the sun is yellow


In [5]:
import chromadb
import hashlib

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_or_create_collection("test_collection", metadata={"hnsw:space": "cosine"})

ids = [hashlib.sha256(sentence.encode()).hexdigest() for sentence in sentences]

collection.upsert(
    ids=ids,
    documents=sentences,
    # metadatas=metadatas,
    embeddings=embeddings,
)

test_query = "reading and writing files in Python"
test_embedding = model.encode(test_query)

results = collection.query(
    query_embeddings=test_embedding,
    n_results=3,
)

for i, result in enumerate(results["documents"]):
    print(f"{i+1}. {result}")

1. ['How do I read a file in Python?', 'Python file I/O tutorial', 'Open a file using the open() function']
